# Validation curves

Validation error and validation loss for every architecture that finished a run. Figures are written to `results/leaderboard/`.


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt


def find_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "coco_mlc").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (src/coco_mlc). Open this notebook from the repository.")


REPO = find_repo()
LEADERBOARD = REPO / "results" / "leaderboard"
LEADERBOARD.mkdir(parents=True, exist_ok=True)

DISPLAY = {
    "shufflenet_v2_x1_0": "ShuffleNet V2 ×1.0",
    "mobilenet_v3_large": "MobileNetV3-Large",
    "efficientnet_b0": "EfficientNet-B0",
    "resnet18": "ResNet18",
    "resnet50": "ResNet50",
    "efficientnet_b4": "EfficientNet-B4",
    "convnext_tiny": "ConvNeXt-Tiny",
    "swin_t": "Swin-T",
    "maxvit_t": "MaxViT-T",
    "efficientnet_v2_s": "EfficientNetV2-S",
    "vgg16": "VGG16",
}


def load_column(path: Path, column: str):
    with path.open(newline="", encoding="utf-8") as handle:
        return [float(row[column]) for row in csv.DictReader(handle)]


def collect_curves(root: Path):
    curves = []
    if not root.is_dir():
        return curves
    for model_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        baseline = model_dir / "history_baseline.csv"
        finetune = model_dir / "history_finetune.csv"
        if not baseline.is_file() or not finetune.is_file():
            continue
        curves.append(
            {
                "name": DISPLAY.get(model_dir.name, model_dir.name),
                "error": load_column(baseline, "val_error") + load_column(finetune, "val_error"),
                "loss": load_column(baseline, "val_loss") + load_column(finetune, "val_loss"),
                "n_baseline": len(load_column(baseline, "val_error")),
            }
        )
    curves.sort(key=lambda row: row["name"])
    return curves


def draw(curves, column, ylabel, title, filename):
    boundary = curves[0]["n_baseline"]
    if any(row["n_baseline"] != boundary for row in curves):
        raise RuntimeError("Baseline length differs across architectures; the stage line would be misleading.")
    fig, ax = plt.subplots(figsize=(12, 7))
    for row in curves:
        y = row[column]
        ax.plot(range(1, len(y) + 1), y, marker="o", markersize=3.5, linewidth=1.6, label=row["name"])
    ax.axvline(boundary + 0.5, color="0.45", linestyle="--", linewidth=1)
    ax.text(
        boundary + 0.65,
        0.98,
        "fine-tuning",
        transform=ax.get_xaxis_transform(),
        ha="left",
        va="top",
        fontsize=9,
        color="0.35",
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, max(len(row[column]) for row in curves) + 1))
    ax.grid(linestyle=":", alpha=0.65)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    fig.tight_layout()
    out = LEADERBOARD / filename
    fig.savefig(out, dpi=140, bbox_inches="tight")
    print("Saved", out.relative_to(REPO))
    plt.show()


## Experiment 1

Epochs 1–5 are the frozen backbone. Epochs 6–13 are fine-tuning.


In [ ]:
curves = collect_curves(REPO / "outputs" / "notebooks")
if not curves:
    print("No finished run for experiment 1.")
else:
    draw(curves, "error", "Error (%)", "Validation error — experiment 1", "experiment_1_validation_error.png")
    draw(curves, "loss", "Loss", "Validation loss — experiment 1", "experiment_1_validation_loss.png")


## Experiment 2

Epochs 1–10 are the frozen backbone. Epochs 11–20 are fine-tuning.


In [ ]:
curves = collect_curves(REPO / "outputs" / "notebooks" / "2")
if not curves:
    print("No finished run for experiment 2.")
else:
    draw(curves, "error", "Error (%)", "Validation error — experiment 2", "experiment_2_validation_error.png")
    draw(curves, "loss", "Loss", "Validation loss — experiment 2", "experiment_2_validation_loss.png")
